In [61]:
import pandas as pd
import json
import numpy as np
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler,StandardScaler

current_dir = Path.cwd()
project_root = current_dir.parents[2]



with open(project_root/"SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/Domain_data.json", "r") as archivo:
    full_domain = json.load(archivo)

with open(project_root/"SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS_Domain_data.json", "r") as archivo:
    updrs_domain = json.load(archivo)

X_multiples= { 'X_STATS':project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS.csv', 
                    'X_V06+STATS': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+STATS.csv',
                      'X_V06+DELTA': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+Deltas.csv'}

y_multiples = { 'target_HY3': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3.csv',
                'target_HY4': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4.csv',
                'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY43.csv',
                'target_MCID': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID.csv'}

csv_output_paths = {'UPDRS':{'FULL_SET':{'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY43/FULL_SET/'},

                                'FEATURE_SELECTION':{'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY43/FEATURE_SELECTION/'}},

                    'FULL':{'FULL_SET':{'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY43/FULL_SET/'},
                                
                                'FEATURE_SELECTION':{'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY43/FEATURE_SELECTION/'}}}

dominios_full = {
    'X_STATS': {
        'full': full_domain['SC_data'] + full_domain['M_data_STATS'] + full_domain['NM_data_STATS'],
        'motor': full_domain['M_data_STATS'],
        'non_motor': full_domain['NM_data_STATS']
    },

    'X_V06+STATS': {
        'full': full_domain['SC_data']
                + full_domain['M_data_V06'] + full_domain['M_data_STATS']
                + full_domain['NM_data_V06'] + full_domain['NM_data_STATS'],
        'motor': full_domain['M_data_V06'] + full_domain['M_data_STATS'],
        'non_motor': full_domain['NM_data_V06'] + full_domain['NM_data_STATS']
    },

    'X_V06+DELTA': {
        'full': full_domain['SC_data']
                + full_domain['M_data_V06'] + full_domain['M_data_DELTA']
                + full_domain['NM_data_V06'] + full_domain['NM_data_DELTA'],
        'motor': full_domain['M_data_V06'] + full_domain['M_data_DELTA'],
        'non_motor': full_domain['NM_data_V06'] + full_domain['NM_data_DELTA']
    }
}

dominios_updrs = {
    'X_STATS': {
        'full': updrs_domain['SC_data']
                + updrs_domain['UPDRS_IMPACTO_MOTOR_STATS']
                + updrs_domain['UPDRS_NO_MOTOR_STATS']
                + updrs_domain['UPDRS_MOTOR_STATS'],
        'examen_motor': updrs_domain['UPDRS_MOTOR_STATS'],
        'impacto_motor': updrs_domain['UPDRS_IMPACTO_MOTOR_STATS'],
        'non_motor': updrs_domain['UPDRS_NO_MOTOR_STATS']
    },

    'X_V06+STATS': {
        'full': updrs_domain['SC_data']
                + updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_STATS']
                + updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_STATS']
                + updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_STATS'],
        'examen_motor': updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_STATS'],
        'impacto_motor': updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_STATS'],
        'non_motor': updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_STATS']
    },

    'X_V06+DELTA': {
        'full': updrs_domain['SC_data']
                + updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_delta']
                + updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_delta']
                + updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_delta'],
        'examen_motor': updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_delta'],
        'impacto_motor': updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_delta'],
        'non_motor': updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_delta']
    }
}



classification_models = {

    "decision_tree": DecisionTreeClassifier(random_state=42,class_weight="balanced"),

    "random_forest": RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),

    "extra_trees": ExtraTreesClassifier(
        random_state=42,
        n_jobs=-1
    ),

    "xgboost": XGBClassifier(
        tree_method="hist",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    ),


    "adaboost": AdaBoostClassifier(
        algorithm="SAMME", 
        random_state=42
    ),

    "svm": SVC(
        kernel="rbf",
        probability=True,  
        random_state=42
    ),

    "logistic_regression": LogisticRegression(
        random_state=42,
        n_jobs=-1,
        max_iter=10000,
        class_weight="balanced"
    ),

    "knn": KNeighborsClassifier(
        n_jobs=-1
    ),

    "gaussian_nb": GaussianNB()
}

In [62]:
import numpy as np
import pandas as pd

from itertools import combinations
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import mutual_info_classif, chi2
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler


# =========================================================
# Feature selectors
# =========================================================

class SpearmanCorrelationDiscard(BaseEstimator, TransformerMixin):
    """
    Elimina variables altamente correlacionadas según Spearman,
    conservando la que tenga mayor correlación absoluta con el target.
    """
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.vars_to_drop_ = None
        self.feature_names_in_ = None

    def fit(self, X, y=None):
        if y is None:
            raise ValueError("SpearmanCorrelationDiscard requiere y en fit().")

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        self.feature_names_in_ = X.columns.to_list()

        data = X.copy()
        data["_target_"] = y

        corr_df = data.corr(method="spearman")

        mask = np.tril(np.ones(corr_df.shape), k=-1).astype(bool)

        corr_long = (
            corr_df.where(mask)
            .stack()
            .reset_index()
            .rename(columns={"level_0": "V1", "level_1": "V2", 0: "CORR"})
        )

        corr_long = corr_long[
            (corr_long["V1"] != "_target_") & (corr_long["V2"] != "_target_")
        ].copy()

        target_corr = corr_df["_target_"]

        corr_long["V1target"] = corr_long["V1"].map(target_corr)
        corr_long["V2target"] = corr_long["V2"].map(target_corr)

        corr_long["WORST_VAR"] = np.where(
            abs(corr_long["V1target"]) <= abs(corr_long["V2target"]),
            corr_long["V1"],
            corr_long["V2"]
        )

        discard_corr_long = corr_long.loc[corr_long["CORR"].abs() > self.threshold]
        self.vars_to_drop_ = list(set(discard_corr_long["WORST_VAR"]))

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.vars_to_drop_, errors="ignore")

        X_df = pd.DataFrame(X, columns=self.feature_names_in_)
        X_df = X_df.drop(columns=self.vars_to_drop_, errors="ignore")
        return X_df


class MIThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01, random_state=42):
        self.threshold = threshold
        self.random_state = random_state
        self.support_ = None
        self.mi_scores_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.mi_scores_ = mutual_info_classif(X_fit, y, random_state=self.random_state)
        self.support_ = self.mi_scores_ >= self.threshold

        if not np.any(self.support_):
            self.support_[np.argmax(self.mi_scores_)] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


class Chi2ThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.0):
        self.threshold = threshold
        self.support_ = None
        self.chi2_scores_ = None
        self.pvalues_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.chi2_scores_, self.pvalues_ = chi2(X_fit, y)
        self.support_ = self.chi2_scores_ >= self.threshold

        if not np.any(self.support_):
            self.support_[np.argmax(self.chi2_scores_)] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


class Chi2MIUnionTopKSelector(BaseEstimator, TransformerMixin):
    """
    Selecciona la unión entre:
    - top_k variables según chi2
    - top_k variables según mutual information

    Nota:
    - chi2 requiere valores no negativos, por eso este selector debe usarse
      después de un MinMaxScaler.
    """
    def __init__(self, top_k=100, random_state=42):
        self.top_k = top_k
        self.random_state = random_state
        self.feature_names_in_ = None
        self.chi2_scores_ = None
        self.mi_scores_ = None
        self.selected_features_ = None
        self.support_ = None
        self.n_features_selected_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        n_features = X_fit.shape[1]
        k = min(self.top_k, n_features)

        self.chi2_scores_, _ = chi2(X_fit, y)
        self.mi_scores_ = mutual_info_classif(X_fit, y, random_state=self.random_state)

        top_chi2_idx = np.argsort(self.chi2_scores_)[::-1][:k]
        top_mi_idx = np.argsort(self.mi_scores_)[::-1][:k]

        selected_idx = np.union1d(top_chi2_idx, top_mi_idx)

        self.support_ = np.zeros(n_features, dtype=bool)
        self.support_[selected_idx] = True
        self.n_features_selected_ = int(self.support_.sum())

        if self.feature_names_in_ is not None:
            self.selected_features_ = [self.feature_names_in_[i] for i in selected_idx]
        else:
            self.selected_features_ = selected_idx.tolist()

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


# =========================================================
# Función principal
# =========================================================

def evaluate_models_pairwise_10x10_oof_and_test(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    scaler=StandardScaler(),
    selectors=None,  # None o lista, ej: [None, "spearman_corr", "mutual_info"]
    sampler_name: str = None,   # None, "smote", "randomoversampling"
    target_original_class_for_sampling: int = 1,
    target_count_for_sampling: int = 180,
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.3,
    random_state: int = 42,
    decimals: int = 4,
    spearman_threshold: float = 0.9,
    mi_threshold: float = 0.01,
    chi2_threshold: float = 0.0,
    chi2_mi_top_k: int = 100,
):
    """
    Pairwise binario con opción de:
    - oversampling
    - feature selection

    Reglas:
    - Si la comparación contiene la clase original 1:
        * la clase 1 se recodifica como positiva (1)
        * la otra clase se recodifica como negativa (0)
        * se aplica oversampling SOLO a la clase positiva hasta target_count_for_sampling
    - Si la comparación NO contiene la clase original 1 (ej. 0 vs 2):
        * no se aplica oversampling
    """

    if selectors is None:
        selectors = [None]

    y_all = y_df.iloc[:, 0].to_numpy()
    X_all = X_df.copy()

    classes = np.sort(np.unique(y_all))
    class_pairs = list(combinations(classes, 2))

    def recode_pairwise_labels(y_pair_original, class_a, class_b):
        if target_original_class_for_sampling in [class_a, class_b]:
            y_bin = np.where(y_pair_original == target_original_class_for_sampling, 1, 0)
            positive_original_class = target_original_class_for_sampling
            negative_original_class = class_b if class_a == target_original_class_for_sampling else class_a
        else:
            y_bin = np.where(y_pair_original == class_a, 0, 1)
            negative_original_class = class_a
            positive_original_class = class_b

        return y_bin, negative_original_class, positive_original_class

    def resolve_sampling_strategy(y_train_fold, positive_original_class):
        if sampler_name is None:
            return None

        if positive_original_class != target_original_class_for_sampling:
            return None

        positive_count = np.sum(y_train_fold == 1)

        if positive_count >= target_count_for_sampling:
            return None

        return {1: target_count_for_sampling}

    def build_sampler(y_train_fold, positive_original_class):
        sampling_strategy = resolve_sampling_strategy(y_train_fold, positive_original_class)

        if sampler_name is None or sampling_strategy is None:
            return None

        sampler_lower = sampler_name.lower()

        if sampler_lower == "randomoversampling":
            return RandomOverSampler(
                sampling_strategy=sampling_strategy,
                random_state=random_state
            )

        elif sampler_lower == "smote":
            
            return SMOTE(
                sampling_strategy=sampling_strategy,
                k_neighbors=3,  # Usamos k=3 para evitar problemas con clases muy pequeñas
                random_state=random_state
            )

        else:
            raise ValueError(
                "sampler_name debe ser None, 'smote' o 'randomoversampling'"
            )

    def build_selector(selector_name):
        if selector_name is None:
            return None

        if selector_name == "spearman_corr":
            return SpearmanCorrelationDiscard(threshold=spearman_threshold)


        elif selector_name == "mutual_info":
            return MIThresholdSelector(
                threshold=mi_threshold,
                random_state=random_state
            )

        elif selector_name == "chi2":
            return Chi2ThresholdSelector(threshold=chi2_threshold)

        elif selector_name == "chi2_mi_union_topk":
            return Chi2MIUnionTopKSelector(
                top_k=chi2_mi_top_k,
                random_state=random_state
            )

        else:
            raise ValueError(f"Selector no soportado: {selector_name}")

    def build_pipeline(estimator, y_train_fold, positive_original_class, selector_name):
        steps = []

        # Selectores que requieren no negativos antes de fit
        if selector_name in ["chi2", "chi2_mi_union_topk"]:
            steps.append(("minmax_before_selector", MinMaxScaler()))

        selector = build_selector(selector_name)
        if selector is not None:
            steps.append(("selector", selector))

        # Escalado general antes del sampler/modelo
        steps.append(("scaler", scaler))

        sampler = build_sampler(y_train_fold, positive_original_class)
        if sampler is not None:
            steps.append(("sampler", sampler))

        steps.append(("model", clone(estimator)))

        return Pipeline(steps)

    def get_positive_class_proba(model, X_input):
        proba = model.predict_proba(X_input)
        class_index = list(model.named_steps["model"].classes_).index(1)
        return proba[:, class_index]

    def compute_metrics(y_true, y_pred, y_proba):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "AUC": roc_auc_score(y_true, y_proba),
        }

    def summarize(metrics_list, suffix):
        df = pd.DataFrame(metrics_list)
        mean = df.mean()
        std = df.std(ddof=1)
        return {
            f"{col}_{suffix}": f"{mean[col]:.{decimals}f} ± {std[col]:.{decimals}f}"
            for col in df.columns
        }

    results = []

    for class_a, class_b in class_pairs:
        print(f"\nEvaluando comparación {class_a} vs {class_b}...")

        mask = np.isin(y_all, [class_a, class_b])
        X_pair = X_all.loc[mask].copy()
        y_pair_original = y_all[mask]

        y, negative_original_class, positive_original_class = recode_pairwise_labels(
            y_pair_original, class_a, class_b
        )

        outer = StratifiedShuffleSplit(
            n_splits=outer_splits,
            test_size=test_size,
            random_state=random_state
        )

        for model_name, estimator in models.items():
            for selector_name in selectors:
                print(f"  Modelo: {model_name} | Selector: {selector_name}")

                test_metrics_all = []
                cv_metrics_all = []

                for split_idx, (train_idx, test_idx) in enumerate(outer.split(X_pair, y), start=1):
                    X_train = X_pair.iloc[train_idx].copy()
                    X_test = X_pair.iloc[test_idx].copy()
                    y_train = y[train_idx]
                    y_test = y[test_idx]

                    unique_train, counts_train = np.unique(y_train, return_counts=True)
                    min_class_count_train = counts_train.min()
                    effective_inner_splits = min(inner_splits, min_class_count_train)

                    if effective_inner_splits < 2:
                        raise ValueError(
                            f"No hay suficientes muestras por clase en train para "
                            f"StratifiedKFold en la comparación {class_a} vs {class_b}."
                        )

                    inner = StratifiedKFold(
                        n_splits=effective_inner_splits,
                        shuffle=True,
                        random_state=random_state
                    )

                    oof_pred = np.zeros(len(y_train), dtype=int)
                    oof_proba = np.zeros(len(y_train), dtype=float)

                    for tr_idx, val_idx in inner.split(X_train, y_train):
                        X_tr = X_train.iloc[tr_idx].copy()
                        X_val = X_train.iloc[val_idx].copy()
                        y_tr = y_train[tr_idx]

                        model = build_pipeline(
                            estimator=estimator,
                            y_train_fold=y_tr,
                            positive_original_class=positive_original_class,
                            selector_name=selector_name
                        )
                        model.fit(X_tr, y_tr)

                        oof_pred[val_idx] = model.predict(X_val)
                        oof_proba[val_idx] = get_positive_class_proba(model, X_val)

                    cv_metrics = compute_metrics(y_train, oof_pred, oof_proba)
                    cv_metrics_all.append(cv_metrics)

                    model_full = build_pipeline(
                        estimator=estimator,
                        y_train_fold=y_train,
                        positive_original_class=positive_original_class,
                        selector_name=selector_name
                    )
                    model_full.fit(X_train, y_train)

                    test_pred = model_full.predict(X_test)
                    test_proba = get_positive_class_proba(model_full, X_test)

                    test_metrics = compute_metrics(y_test, test_pred, test_proba)
                    test_metrics_all.append(test_metrics)

                    print(
                        f"    Split {split_idx:02d} | "
                        f"AUC CV: {cv_metrics['AUC']:.{decimals}f} | "
                        f"AUC Test: {test_metrics['AUC']:.{decimals}f}"
                    )

                sampling_applied = (
                    "Sí"
                    if sampler_name is not None and positive_original_class == target_original_class_for_sampling
                    else "No"
                )

                row = {
                    "Comparison": f"{class_a} vs {class_b}",
                    "Negative_Class": negative_original_class,
                    "Positive_Class": positive_original_class,
                    "Model": model_name,
                    "Selector": selector_name if selector_name is not None else "None",
                    "Sampler": sampler_name if sampler_name is not None else "None",
                    "Sampling_Applied": sampling_applied,
                    "Sampling_Target": (
                        f"Clase original {target_original_class_for_sampling} -> {target_count_for_sampling}"
                        if sampling_applied == "Sí"
                        else "No aplica"
                    ),
                }
                row.update(summarize(test_metrics_all, "Testing"))
                row.update(summarize(cv_metrics_all, "CV"))

                results.append(row)

    df_results = pd.DataFrame(results)

    ordered_cols = [
        "Comparison", "Negative_Class", "Positive_Class", "Model",
        "Selector", "Sampler", "Sampling_Applied", "Sampling_Target",
        "Accuracy_Testing", "Precision_Testing", "Recall_Testing", "F1_Testing", "AUC_Testing",
        "Accuracy_CV", "Precision_CV", "Recall_CV", "F1_CV", "AUC_CV"
    ]

    return df_results[ordered_cols]

# ONE vs REST MODEL

## DEFAULT

In [63]:
for val in dominios_updrs:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY43']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")

        for val2 in dominios_updrs[val]:
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios_updrs[val][val2]))
            X_sub = X[dominios_updrs[val][val2]]
            print(f"      Data shape: {X_sub.shape}")
            for scaler in [StandardScaler(), MinMaxScaler()]:
                print(f"      Evaluating with scaler: {scaler.__class__.__name__}")
                df=evaluate_models_pairwise_10x10_oof_and_test(
                                        X_df=X_sub, 
                                        y_df=y,
                                        sampler_name=None,
                                        selectors=[None],
                                        scaler=scaler, 
                                        models=classification_models, 
                                        random_state=42
                                    )
                print(f"      Saving results to: {csv_output_paths['UPDRS']['FULL_SET'][target]/f'{val}_{val2}_{scaler.__class__.__name__}_ONE_VS_REST.csv'}")
                df.to_csv(csv_output_paths['UPDRS']['FULL_SET'][target]/f"{val}_{val2}_FS_{scaler.__class__.__name__}_{target}_ONE_VS_REST.csv")

Evaluating domain: X_STATS
  Target: target_HY43, Shape: (909, 1)
    Subdomain: full
      Loading data... N. Features: 301
      Data shape: (909, 301)
      Evaluating with scaler: StandardScaler

Evaluando comparación 0 vs 1...
  Modelo: decision_tree | Selector: None
    Split 01 | AUC CV: 0.7302 | AUC Test: 0.8089
    Split 02 | AUC CV: 0.7368 | AUC Test: 0.7715
    Split 03 | AUC CV: 0.7243 | AUC Test: 0.6268
    Split 04 | AUC CV: 0.7877 | AUC Test: 0.7301
    Split 05 | AUC CV: 0.7645 | AUC Test: 0.7882
    Split 06 | AUC CV: 0.7447 | AUC Test: 0.8057
    Split 07 | AUC CV: 0.7675 | AUC Test: 0.7427
    Split 08 | AUC CV: 0.7260 | AUC Test: 0.7297
    Split 09 | AUC CV: 0.7930 | AUC Test: 0.7346
    Split 10 | AUC CV: 0.7429 | AUC Test: 0.7134
  Modelo: random_forest | Selector: None
    Split 01 | AUC CV: 0.9047 | AUC Test: 0.9236
    Split 02 | AUC CV: 0.9000 | AUC Test: 0.9371
    Split 03 | AUC CV: 0.9208 | AUC Test: 0.8701
    Split 04 | AUC CV: 0.8783 | AUC Test: 0.9291


In [65]:
for val in dominios_full:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY43']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")

        for val2 in dominios_full[val]:
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios_full[val][val2]))
            X_sub = X[dominios_full[val][val2]]
            print(f"      Data shape: {X_sub.shape}")
            for scaler in [StandardScaler(), MinMaxScaler()]:
                print(f"      Evaluating with scaler: {scaler.__class__.__name__}")
                df=evaluate_models_pairwise_10x10_oof_and_test(
                                        X_df=X_sub, 
                                        y_df=y,
                                        sampler_name=None,
                                        selectors=[None],
                                        scaler=scaler, 
                                        models=classification_models, 
                                        random_state=42
                                    )
                print(f"      Saving results to: {csv_output_paths['FULL']['FULL_SET'][target]/f'{val}_{val2}_{scaler.__class__.__name__}_ONE_VS_REST.csv'}")
                df.to_csv(csv_output_paths['FULL']['FULL_SET'][target]/f"{val}_{val2}_FS_{scaler.__class__.__name__}_{target}_ONE_VS_REST.csv")

Evaluating domain: X_STATS
  Target: target_HY43, Shape: (909, 1)
    Subdomain: full
      Loading data... N. Features: 931
      Data shape: (909, 931)
      Evaluating with scaler: StandardScaler

Evaluando comparación 0 vs 1...
  Modelo: decision_tree | Selector: None
    Split 01 | AUC CV: 0.7550 | AUC Test: 0.7427
    Split 02 | AUC CV: 0.7065 | AUC Test: 0.6724
    Split 03 | AUC CV: 0.7192 | AUC Test: 0.6931
    Split 04 | AUC CV: 0.7082 | AUC Test: 0.6593
    Split 05 | AUC CV: 0.7320 | AUC Test: 0.7301
    Split 06 | AUC CV: 0.7407 | AUC Test: 0.8350
    Split 07 | AUC CV: 0.7552 | AUC Test: 0.7256
    Split 08 | AUC CV: 0.7313 | AUC Test: 0.7553
    Split 09 | AUC CV: 0.7480 | AUC Test: 0.7720
    Split 10 | AUC CV: 0.7065 | AUC Test: 0.7093
  Modelo: random_forest | Selector: None
    Split 01 | AUC CV: 0.8859 | AUC Test: 0.9221
    Split 02 | AUC CV: 0.8884 | AUC Test: 0.8993
    Split 03 | AUC CV: 0.9116 | AUC Test: 0.8809
    Split 04 | AUC CV: 0.8749 | AUC Test: 0.9122


KeyboardInterrupt: 

# OVERSAMPLING METHOD

In [64]:
for val in dominios_updrs:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY43']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")

        for val2 in dominios_updrs[val]:
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios_updrs[val][val2]))
            X_sub = X[dominios_updrs[val][val2]]
            print(f"      Data shape: {X_sub.shape}")
            for sampler in ["smote", "randomoversampling"]:
                print(f"      Evaluating with sampler: {sampler}")
                df=evaluate_models_pairwise_10x10_oof_and_test(
                                        X_df=X_sub, 
                                        y_df=y,
                                        sampler_name=sampler,
                                        selectors=[None],
                                        scaler=MinMaxScaler(), 
                                        models=classification_models, 
                                        random_state=42
                                    )
                print(f"      Saving results to: {csv_output_paths['UPDRS']['FULL_SET'][target]/f'{val}_{val2}_FS_{sampler}_ONE_VS_REST.csv'}")
                df.to_csv(csv_output_paths['UPDRS']['FULL_SET'][target]/f"{val}_{val2}_{sampler}_{target}_ONE_VS_REST.csv")

Evaluating domain: X_STATS
  Target: target_HY43, Shape: (909, 1)
    Subdomain: full
      Loading data... N. Features: 301
      Data shape: (909, 301)
      Evaluating with sampler: smote

Evaluando comparación 0 vs 1...
  Modelo: decision_tree | Selector: None
    Split 01 | AUC CV: 0.6937 | AUC Test: 0.7386
    Split 02 | AUC CV: 0.7555 | AUC Test: 0.7215
    Split 03 | AUC CV: 0.7572 | AUC Test: 0.6976
    Split 04 | AUC CV: 0.7445 | AUC Test: 0.7841
    Split 05 | AUC CV: 0.7247 | AUC Test: 0.8049
    Split 06 | AUC CV: 0.7027 | AUC Test: 0.7764
    Split 07 | AUC CV: 0.7210 | AUC Test: 0.7012
    Split 08 | AUC CV: 0.7640 | AUC Test: 0.7675
    Split 09 | AUC CV: 0.7315 | AUC Test: 0.7886
    Split 10 | AUC CV: 0.7658 | AUC Test: 0.7350
  Modelo: random_forest | Selector: None
    Split 01 | AUC CV: 0.9010 | AUC Test: 0.9075
    Split 02 | AUC CV: 0.8968 | AUC Test: 0.9251
    Split 03 | AUC CV: 0.9118 | AUC Test: 0.8751
    Split 04 | AUC CV: 0.8519 | AUC Test: 0.9431
    Spli

In [ ]:
for val in dominios_full:
    print(f"Evaluating domain: {val}")
    path=X_multiples[val]
    X=pd.read_csv(path, index_col=0)
    for target in ['target_HY43']:
        path=y_multiples[target]
        y=pd.read_csv(path, index_col=0)
        print(f"  Target: {target}, Shape: {y.shape}")

        for val2 in dominios_full[val]:
            print(f"    Subdomain: {val2}")
            print("      Loading data...", 'N. Features:', len(dominios_full[val][val2]))
            X_sub = X[dominios_full[val][val2]]
            print(f"      Data shape: {X_sub.shape}")
            for sampler in ["smote", "randomoversampling"]:
                print(f"      Evaluating with sampler: {sampler}")
                df=evaluate_models_pairwise_10x10_oof_and_test(
                                        X_df=X_sub, 
                                        y_df=y,
                                        sampler_name=sampler,
                                        selectors=[None],
                                        scaler=MinMaxScaler(), 
                                        models=classification_models, 
                                        random_state=42
                                    )
                print(f"      Saving results to: {csv_output_paths['FULL']['FULL_SET'][target]/f'{val}_{val2}_{sampler}_ONE_VS_REST.csv'}")
                df.to_csv(csv_output_paths['FULL']['FULL_SET'][target]/f"{val}_{val2}_{sampler}_{target}_ONE_VS_REST.csv")